# LLM Support Training and Evaluation

This notebook will load the prepared datasets, evaluate a base open-weight LLM, personalize it, and compare both versions on the test set.


## Install dependencies

Install the project requirements in the active notebook environment before running the rest of the notebook.


In [ ]:
%pip install -r requirements.txt


## Imports

Add the libraries needed for loading datasets, running the model, and computing evaluation metrics.


In [3]:
import json
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType
from support_classification.evaluation import evaluate_model, predict_queue
from support_classification.prompts import build_prompt
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from support_classification.paths import LABELS_PATH, PROJECT_ROOT, TRAIN_DATASET_PATH, TEST_DATASET_PATH

# Display long prompt text without truncation when inspecting the datasets.
pd.set_option('display.max_colwidth', 120)


ModuleNotFoundError: No module named 'torch'

## Load prepared datasets

Load the train and test CSV files produced by the preparation notebook.


In [3]:
train_dataset = pd.read_csv(TRAIN_DATASET_PATH)
test_dataset = pd.read_csv(TEST_DATASET_PATH)
valid_labels = json.loads(LABELS_PATH.read_text(encoding='utf-8'))['labels']

print('Train shape:', train_dataset.shape)
print('Test shape:', test_dataset.shape)
print('Valid labels:', valid_labels)


Train shape: (479, 2)
Test shape: (120, 2)


## Load the base model

Load the chosen open-weight model and prepare it for the personalization step.


In [ ]:
# Initial model choice kept here as a fallback option.
# Correct official id: mistralai/Ministral-3-8B-Instruct-2512
# This open-weight instruct model is multilingual and strong enough for support-ticket classification,
# so it remains a good backup if the smaller model does not reach the target score later.
# base_model_id = 'mistralai/Ministral-3-8B-Instruct-2512'

# Start with a smaller instruct model because the training set is modest (479 examples).
# Qwen2.5-1.5B-Instruct is much lighter to fine-tune, cheaper to run, and still suitable for multilingual text classification.
base_model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Define a padding token for batching during training.
# Reuse the EOS token because many causal LLM tokenizers do not provide a pad token by default.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)

# Keep the model configuration aligned with the tokenizer padding behavior.
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

# Keep special-token ids aligned even when some of them are undefined on the tokenizer.
# This avoids later warnings when training utilities resynchronize model and tokenizer configs.
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

# Disable the KV cache during training to avoid incompatibilities with gradient checkpointing or fine-tuning utilities.
model.config.use_cache = False

print('Loaded model:', base_model_id)


## Define the inference prompt

Reuse the same input structure as in the preparation notebook so the model sees consistent prompts.


In [ ]:
# Prompt helpers are imported from support_classification.prompts so the training notebook uses the exact
# same template as the data-preparation notebook.
# The processed datasets already contain this prompt format, and this example shows how to
# rebuild it later for new tickets at inference time.
example_prompt = build_prompt(
    subject='Compatibility Inquiry',
    body='I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.',
    language='en',
    business_type='Tech Online Store',
    valid_labels=valid_labels,
)

print(example_prompt)


You are a support ticket classification assistant.
Predict the ticket category from the information below.

Subject: Compatibility Inquiry
Body: I recently purchased a MacBook Air M1 and need to know if it is compatible with my external wireless monitor setup.
Language: en
Business type: Tech Online Store

Answer:


## Prepare fine-tuning data

Format the training dataset so it can be used by the personalization pipeline in the next step.


In [ ]:
# Keep only the training split for the personalization step.
# The test set remains untouched and will only be used later during evaluation.
train_sft_dataset = train_dataset.copy()

# Build a single text field for supervised fine-tuning while keeping the original columns unchanged.
# The prompt already ends with 'Answer:', so we prepend a space before the expected label.
train_sft_dataset['text'] = train_sft_dataset.apply(
    lambda row: f"{row['prompt']} {row['answer']}",
    axis=1,
)

# Convert the prepared DataFrame to a Hugging Face Dataset, which is the format expected by SFTTrainer.
hf_train_sft_dataset = Dataset.from_pandas(
    train_sft_dataset[['text']],
    preserve_index=False,
)

display(train_sft_dataset[['prompt', 'answer', 'text']].head(2))
print(hf_train_sft_dataset)


## Fine-tune with LoRA

Configure a lightweight LoRA adaptation on the training split and save the personalized model artifacts.


In [ ]:
# A small LoRA setup is enough for this project and keeps training affordable.
# The chosen target modules match the main projection layers used by Qwen2.5 causal language models.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

# Save the personalized adapter in the project so it can be reused later without retraining.
adapter_output_dir = PROJECT_ROOT / 'artifacts' / 'qwen25-support-lora'
adapter_output_dir.mkdir(parents=True, exist_ok=True)

# Prefer bf16 on supported GPUs, otherwise fall back to fp16 on CUDA and fp32 on CPU.
bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled

# Enable gradient checkpointing to reduce memory usage during fine-tuning.
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

# Training hyperparameters are intentionally simple and conservative for a small supervised dataset.
sft_config = SFTConfig(
    output_dir=str(adapter_output_dir),
    dataset_text_field='text',
    max_length=768,
    packing=False,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    gradient_checkpointing=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    bf16=bf16_enabled,
    fp16=fp16_enabled,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=hf_train_sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(str(adapter_output_dir))

# Save the tokenizer with the LoRA adapter so the fine-tuned model can be reloaded later
# with the same tokenization settings.
tokenizer.save_pretrained(str(adapter_output_dir))

print('Saved personalized model artifacts to:', adapter_output_dir)


## Verify saved artifacts

Check that the LoRA adapter and tokenizer files were saved correctly after training.


In [ ]:
# List the saved files so we can confirm that the adapter and tokenizer artifacts are available.
saved_artifacts = sorted(path.name for path in adapter_output_dir.iterdir())
print('Saved artifacts:')
for artifact_name in saved_artifacts:
    print('-', artifact_name)


## Run a qualitative check

Reload the personalized adapter and inspect a few predictions as a quick sanity check to confirm that the saved model reloads correctly and produces clean labels before the full quantitative evaluation on the test set.


In [ ]:
# Reload the tokenizer and the trained adapter from disk to verify that the saved artifacts are reusable.
inference_tokenizer = AutoTokenizer.from_pretrained(str(adapter_output_dir))
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
personalized_model = PeftModel.from_pretrained(base_model_for_inference, str(adapter_output_dir))
personalized_model.eval()

# Inspect a few training examples to confirm that the personalized model produces a clean label.
qualitative_examples = train_dataset.head(3).copy()
qualitative_examples['prediction'] = qualitative_examples['prompt'].apply(
    lambda prompt: predict_queue(prompt, personalized_model, inference_tokenizer, valid_labels)
)
display(qualitative_examples[['prompt', 'answer', 'prediction']])


## Evaluate both models

Run the same test procedure on the shared test split and compute the weighted F1-score for both versions.


In [ ]:
# Reload a fresh copy of the base model so we can compare it fairly against the personalized adapter.
base_model_for_evaluation = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    low_cpu_mem_usage=True,
)
base_model_for_evaluation.eval()

base_test_predictions, base_weighted_f1 = evaluate_model(
    test_dataset,
    base_model_for_evaluation,
    tokenizer,
    valid_labels,
    'Base model',
)

personalized_test_predictions, personalized_weighted_f1 = evaluate_model(
    test_dataset,
    personalized_model,
    inference_tokenizer,
    valid_labels,
    'Personalized model',
)

display(personalized_test_predictions[['prompt', 'answer', 'prediction']].head(5))


## Compare results

Summarize the base and personalized model scores in a compact comparison table.


In [ ]:
target_f1 = 0.92

comparison_df = pd.DataFrame(
    [
        {
            'model': 'Base model',
            'weighted_f1': base_weighted_f1,
            'target_92_reached': base_weighted_f1 >= target_f1,
        },
        {
            'model': 'Personalized model',
            'weighted_f1': personalized_weighted_f1,
            'target_92_reached': personalized_weighted_f1 >= target_f1,
        },
    ]
)
comparison_df['weighted_f1'] = comparison_df['weighted_f1'].map(lambda score: f'{score:.2%}')

f1_improvement = personalized_weighted_f1 - base_weighted_f1
print(f'Weighted F1 improvement: {f1_improvement:+.2%}')
print(f'Personalized model reached the 92% target: {personalized_weighted_f1 >= target_f1}')
display(comparison_df)
